In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
from datasets import load_dataset

/home/hz/CUDA/resnet18-exp/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms

In [3]:
import sys
import os
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../src'))

In [4]:
from src.models.amp_freq_resnet18 import amp_freq_resnet18_keras

## data

In [5]:
transform_train = transforms.Compose([
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225)),
])

def train_transform(batch):
    batch["image"] = [transform_train(img.convert("RGB")) for img in batch["image"]]
    return batch

def test_transform(batch):
    batch["image"] = [transform_test(img.convert("RGB")) for img in batch["image"]]
    return batch

def collate_fn(batch):
    imgs   = torch.stack([b["image"] for b in batch])  # (B, C, H, W)
    imgs   = imgs.permute(0, 2, 3, 1)                  # (B, H, W, C)
    ohe_labels = F.one_hot(torch.tensor([b["label"] for b in batch]), num_classes=NUM_CLASSES).float()
    return imgs, ohe_labels


def get_dataloaders(batch_size=128):
    dataset = load_dataset("zh-plus/tiny-imagenet", cache_dir="../data/raw")
    dataset["train"].set_transform(train_transform)
    dataset["valid"].set_transform(test_transform)

    trainloader = torch.utils.data.DataLoader(
        dataset["train"], batch_size=batch_size, shuffle=True, pin_memory=True,
        num_workers=6, persistent_workers=True,
        collate_fn=collate_fn)

    testloader = torch.utils.data.DataLoader(
        dataset["valid"], batch_size=batch_size, shuffle=False, pin_memory=True,
        num_workers=6, persistent_workers=True,
        collate_fn=collate_fn)

    return trainloader, testloader

## training

In [6]:
EPOCHS = 30
BATCH_SIZE = 128
LR = 1e-3
NUM_CLASSES = 200

keras.utils.set_random_seed(0)

In [7]:
model = amp_freq_resnet18_keras((64, 64, 3), num_classes=NUM_CLASSES, amp_factors=[0.8, 0.6, 0.4, 0.2], thresh_quantiles=[0.2, 0.4, 0.6, 0.8])

In [8]:
train_loader, val_loader = get_dataloaders(BATCH_SIZE)

In [9]:
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LR),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [10]:
history = model.fit(train_loader, validation_data=val_loader, epochs=EPOCHS, verbose=2)

/home/hz/CUDA/resnet18-exp/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a torch DataLoader. The DataLoader is expected to already be shuffled.
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/30


782/782 - 46s - 58ms/step - accuracy: 0.0846 - loss: 4.4456 - val_accuracy: 0.1298 - val_loss: 4.0245


Epoch 2/30


782/782 - 48s - 62ms/step - accuracy: 0.1845 - loss: 3.6665 - val_accuracy: 0.1883 - val_loss: 3.6384


Epoch 3/30


... (51 outputs left out)

Epoch 29/30


782/782 - 49s - 62ms/step - accuracy: 0.8768 - loss: 0.4006 - val_accuracy: 0.3831 - val_loss: 4.1330


Epoch 30/30


782/782 - 48s - 61ms/step - accuracy: 0.8794 - loss: 0.3872 - val_accuracy: 0.4055 - val_loss: 3.8815


In [11]:
print(f"best val acc: {100 * max(history.history['val_accuracy']):.2f}%")

best val acc: 42.32%
